# ☕ GroupBy y pivoteo: la cafetería

**Taller de Obtención y Preparación de Datos** · batería de práctica del [Visualizador TOPD](https://mati3939.github.io/visualizador-numpy-pandas/)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mati3939/visualizador-numpy-pandas/blob/main/notebooks/08_groupby_pivoteo.ipynb)

**Objetivos:** dominar split-apply-combine: agregaciones, `agg` múltiple, `pivot_table` y porcentajes por grupo.

🔎 **Apóyate en el visualizador:** [GroupBy y pivoteo](https://mati3939.github.io/visualizador-numpy-pandas/#groupby) — ten la pestaña abierta mientras resuelves; cada concepto de aquí está animado allá.

---

## Contexto: cadena de cafeterías en Concepción

Administras 3 sucursales (Centro, San Pedro, Talcahuano). Gerencia quiere respuestas, no tablas crudas — las mismas preguntas de la **Guía de GroupBy** oficial del curso, aquí con autochequeo.

## 0 · Preparación

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 300
ventas = pd.DataFrame({
    'sucursal': rng.choice(['Centro', 'San Pedro', 'Talcahuano'], n, p=[.5, .3, .2]),
    'categoria': rng.choice(['Café', 'Té', 'Pastelería', 'Sándwich'], n, p=[.4, .15, .25, .2]),
    'vendedor': rng.choice(['Amanda', 'Bruno', 'Carla', 'Diego'], n),
    'pago': rng.choice(['efectivo', 'débito', 'crédito'], n, p=[.25, .55, .2]),
    'cantidad': rng.integers(1, 5, n),
    'precio_unit': rng.choice([1800, 2500, 3200, 4500], n),
})
ventas['total'] = ventas['cantidad'] * ventas['precio_unit']
ventas.head()

## 1 · Calentamiento

Ten a mano la animación de split-apply-combine: [https://mati3939.github.io/visualizador-numpy-pandas/#groupby](https://mati3939.github.io/visualizador-numpy-pandas/#groupby)

### 1.1 Facturación por sucursal (pregunta 1 de la guía) ⭐

Calcula `fact`, la facturación total (`total`) de **cada sucursal**, ordenada de mayor a menor.

<details><summary>💡 Pista (haz clic)</summary>

groupby entrega los grupos en orden alfabético — el ranking lo da `sort_values`.

</details>

In [ ]:
fact = ...   # TU CÓDIGO AQUÍ

print(fact)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert len(fact) == 3
assert fact.sum() == ventas['total'].sum(), 'la suma de los grupos debe cuadrar con el total'
assert list(fact) == sorted(fact, reverse=True), 'debe venir ordenada de mayor a menor'
print('✅ ¡Correcto!')

### 1.2 Ticket promedio por medio de pago (pregunta 2) ⭐

Calcula `ticket`, el promedio de `total` según `pago`. ¿Cuál medio de pago tiene el ticket más alto? Guárdalo en `pago_top` (texto).

<details><summary>💡 Pista (haz clic)</summary>

`idxmax()` devuelve la etiqueta del índice donde está el máximo — para un groupby, el nombre del grupo ganador.

</details>

In [ ]:
ticket = ...     # TU CÓDIGO AQUÍ
pago_top = ...

print(ticket.round(0))
print(pago_top)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert set(ticket.index) == {'efectivo', 'débito', 'crédito'}
assert pago_top == ticket.idxmax()
assert isinstance(pago_top, str), 'pago_top es el NOMBRE del medio de pago'
print('✅ ¡Correcto!')

## 2 · Núcleo

### 2.1 count vs size (pregunta 3) ⭐⭐

Calcula `n_ventas`, el número de **transacciones** que atendió cada vendedor. Piensa: ¿`count()` o `size()`? ¿Qué cambiaría si `total` tuviera nulos?

In [ ]:
n_ventas = ...   # TU CÓDIGO AQUÍ

print(n_ventas)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert n_ventas.sum() == len(ventas)
assert len(n_ventas) == 4
print('✅ ¡Correcto!')

### 2.2 Tres respuestas en una tabla (pregunta 6) ⭐⭐

Para cada `categoria` entrega en UNA sola tabla `resumen`: la facturación total, el ticket promedio y el número de ventas — columnas `sum`, `mean` y `count` (en ese orden).

<details><summary>💡 Pista (haz clic)</summary>

`.agg(['sum', 'mean', 'count'])` aplica varias funciones de una pasada.

</details>

In [ ]:
resumen = ...   # TU CÓDIGO AQUÍ

print(resumen.round(0))

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert list(resumen.columns) == ['sum', 'mean', 'count']
assert resumen['sum'].sum() == ventas['total'].sum()
assert resumen['count'].sum() == len(ventas)
print('✅ ¡Correcto!')

### 2.3 La tabla cruzada (pregunta 7) ⭐⭐

Construye `cruce`: facturación total por **sucursal (filas) × categoría (columnas)** con `pd.pivot_table`. Si una combinación no registra ventas debe mostrar **0**, no NaN.

<details><summary>💡 Pista (haz clic)</summary>

El heatmap del visualizador ES esta tabla: https://mati3939.github.io/visualizador-numpy-pandas/#groupby (tarjeta pivot_table).

</details>

In [ ]:
cruce = ...   # TU CÓDIGO AQUÍ

print(cruce)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert cruce.shape == (3, 4)
assert not cruce.isnull().any().any(), 'usa fill_value para los huecos'
assert cruce.to_numpy().sum() == ventas['total'].sum()
print('✅ ¡Correcto!')

## 3 · Desafíos de gerencia

### 3.1 El producto estrella de cada sucursal (pregunta 9) 🏁 ⭐⭐⭐

¿Qué `categoria` vende más **unidades** (`cantidad`) dentro de cada sucursal? Construye `estrella`, una Serie con índice sucursal y valor la categoría ganadora (p. ej. `Centro → Café`).

<details><summary>💡 Pista (haz clic)</summary>

Agrupa por DOS llaves, y luego vuelve a agrupar el resultado solo por sucursal para pedir `idxmax()`.

</details>

In [ ]:
estrella = ...   # TU CÓDIGO AQUÍ

print(estrella)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert len(estrella) == 3
assert set(estrella.index) == {'Centro', 'San Pedro', 'Talcahuano'}
_u = ventas.groupby(['sucursal', 'categoria'])['cantidad'].sum()
for _s in estrella.index:
    assert _u[_s][estrella[_s]] == _u[_s].max(), f'revisa {_s}'
print('✅ ¡Correcto!')

### 3.2 ¿Qué porcentaje aporta cada categoría? (pregunta 10) 🏁 ⭐⭐⭐

A partir de `cruce`, construye `pct` donde cada fila (sucursal) muestre el **porcentaje** que aporta cada categoría a SU facturación. Cada fila debe sumar 100.

<details><summary>💡 Pista (haz clic)</summary>

`cruce.div(cruce.sum(axis=1), axis=0)` — el `axis=0` del div alinea el divisor con las filas.

</details>

In [ ]:
pct = ...   # TU CÓDIGO AQUÍ

print(pct.round(1))

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert pct.shape == cruce.shape
assert (abs(pct.sum(axis=1) - 100) < 1e-6).all(), 'cada fila debe sumar 100'
print('✅ ¡Correcto!')

---

## 🎯 Chequea tu intuición

Antes de cerrar, abre el visualizador y en el botón **🎯 Ejercicios** corre una ronda de [Predice la salida](https://mati3939.github.io/visualizador-numpy-pandas/#predice) y [Detective de bugs](https://mati3939.github.io/visualizador-numpy-pandas/#detective) filtrando por **GroupBy**. Si un error de Python te deja pegado, pégalo en el [Traductor de errores](https://mati3939.github.io/visualizador-numpy-pandas/#traductor).